# 딥앙상블 부트스트랩 v2 로컬 학습 (Windows + AMD RX 9070 XT / ROCm, WSL2)

`finetune_ensemble_v2_local_rtx.ipynb`(NVIDIA/CUDA용)의 ROCm 버전. 이 PC(`C:\fine-tune`)는
실제로는 **NVIDIA RTX가 아니라 AMD RX 9070 XT**였음(§2.22 문서 작성 시점엔 "RTX(집)"으로
적었으나 실제 GPU는 §2.14와 동일한 9070 XT) — 그래서 CUDA 대신 ROCm으로 진행한다.

**중요**: §2.14에서 이미 확인됐듯 **네이티브 Windows ROCm은 이 GPU(gfx1201)에서 MIOpen
JIT 컴파일이 깨짐** — 그래서 이 노트북은 네이티브 Windows에서 돌리는 게 아니라, **WSL2
Ubuntu-22.04 안의 Jupyter 커널**로 열어서 실행하는 걸 전제로 한다(`finetune_twinlitenetplus_
local_windows_rocm.ipynb`가 네이티브 Windows ROCm 휠 설치를 시도하는 것과 다름 — 그 노트북은
§2.14 WSL 전환 이전에 쓰인 것이라 실제로 검증된 경로가 아님, 참고용으로만 남겨둠).

이 PC에는 이미 `~/fine-tune/.venv`(WSL 안)에 ROCm PyTorch(2.9.1+rocm7.2.0)가 설치돼
있고 `torch.cuda.is_available()`이 True로 RX 9070 XT를 인식하는 것까지 확인됨(§2.14에서
40epoch medium 학습 완료 실적 있음) — 0단계(드라이버/ROCm SDK 설치)는 생략하고 바로
이 venv를 재사용한다.

**실행 위치**: WSL2 안에서 `jupyter lab`/`jupyter notebook`을 띄우고 이 파일을 열 것
(`cd ~/fine-tune && source .venv/bin/activate && pip install -q jupyterlab && jupyter lab`).
코드 자체(패치/`finetune.py`)는 CUDA/ROCm/MPS 어디서든 동일하게 동작하도록 짜여 있어서
(§2.17) `finetune_ensemble_v2_local_rtx.ipynb`와 거의 같고, 차이는 0단계(설치)뿐이다.

## 0. 환경 확인 (설치는 이미 돼 있음 - 확인만)

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda(=ROCm/HIP) available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('ROCm 인식 안 됨 - ~/fine-tune/.venv를 activate하고 이 커널을 다시 띄웠는지 확인할 것')

## 1. 로컬 경로 설정

In [ ]:
import os

BASE_DIR = os.path.expanduser('~/fine-tune')  # WSL 쪽 경로 (Windows C:\fine-tune과는 별도 파일시스템)
WORK_DIR = BASE_DIR
REPO_DIR = os.path.join(WORK_DIR, 'TwinLiteNetPlus')
DATA_DIR = os.path.join(WORK_DIR, 'bdd100k')

print('BASE_DIR:', BASE_DIR)
print('REPO_DIR:', REPO_DIR, '(이미 §2.14에서 clone/설치 완료)')
print('bootstrap_v2.zip 있는지:', os.path.isfile(os.path.join(BASE_DIR, 'bootstrap_v2.zip')))
print('pretrained/medium.pth 있는지:', os.path.isfile(os.path.join(REPO_DIR, 'pretrained', 'medium.pth')))

## 2. 데이터 준비 (bootstrap_v2, 1016장, 증강 없음 - Mac ensemble v1 라운드와 동일 컨벤션)

In [ ]:
import glob, random, shutil, zipfile

CONFIG = 'medium'
LOCAL_ZIP = os.path.join(BASE_DIR, 'bootstrap_v2.zip')
LOCAL_BOOTSTRAP = os.path.join(BASE_DIR, 'bootstrap_v2')
if not os.path.isdir(LOCAL_BOOTSTRAP):
    assert os.path.isfile(LOCAL_ZIP), f'{LOCAL_ZIP} 없음'
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        zf.extractall(BASE_DIR)
    print('압축 해제 완료:', LOCAL_BOOTSTRAP)

SRC_IMG = os.path.join(LOCAL_BOOTSTRAP, 'images')
SRC_DA = os.path.join(LOCAL_BOOTSTRAP, 'da_masks')
SRC_LL = os.path.join(LOCAL_BOOTSTRAP, 'll_masks')
VAL_RATIO = 0.15
SEED = 42

names = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(SRC_IMG, '*.png')))
assert names, f'{SRC_IMG}에 png가 없음'

random.Random(SEED).shuffle(names)
n_val = max(1, int(len(names) * VAL_RATIO))
val_names, train_names = names[:n_val], names[n_val:]
print(f'전체 {len(names)}장 -> train {len(train_names)} / val {len(val_names)}')

if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
for split, split_names in [('train', train_names), ('val', val_names)]:
    for sub in ['images', 'drivable_area_annotations', 'lane_line_annotations']:
        os.makedirs(os.path.join(DATA_DIR, sub, split), exist_ok=True)
    for n in split_names:
        shutil.copy(os.path.join(SRC_IMG, n + '.png'), os.path.join(DATA_DIR, 'images', split, n + '.png'))
        shutil.copy(os.path.join(SRC_DA, n + '.png'), os.path.join(DATA_DIR, 'drivable_area_annotations', split, n + '.png'))
        shutil.copy(os.path.join(SRC_LL, n + '.png'), os.path.join(DATA_DIR, 'lane_line_annotations', split, n + '.png'))
print('정리 완료:', DATA_DIR)

## 3. 하이퍼파라미터 + 코드 패치 (`finetune_ensemble_v2_local_rtx.ipynb` cell-14와 완전히 동일한 로직, 멱등)

In [ ]:
import yaml

with open(f'{REPO_DIR}/hyperparameters/twinlitev2_hyper.yaml') as f:
    hyp = yaml.safe_load(f)
hyp['lr'] = hyp['lr'] * 0.1
hyp['prob_crop'] = 0.0
hyp['ll_loss_weight'] = 2.0
hyp['ll_lr_mult'] = 2.5
FINETUNE_HYP = f'{REPO_DIR}/hyperparameters/finetune_hyper.yaml'
with open(FINETUNE_HYP, 'w') as f:
    yaml.safe_dump(hyp, f)
print('저장:', FINETUNE_HYP)

loss_path = f'{REPO_DIR}/loss.py'
with open(loss_path) as f:
    loss_src = f.read()
if 'self.ll_weight' in loss_src:
    print('loss.py: ll_weight 이미 패치됨 -> 스킵')
else:
    old_init = '''        self.seg_tver_da = TverskyLoss(mode="multiclass", alpha=alpha1, beta=1-alpha1, gamma=gamma1, from_logits=True)
        self.seg_tver_ll = TverskyLoss(mode="multiclass", alpha=alpha2, beta=1-alpha2, gamma=gamma2, from_logits=True)
        self.seg_focal = FocalLossSeg(mode="multiclass", alpha=alpha3, gamma=gamma3)'''
    new_init = old_init + '\n        self.ll_weight = hyp.get("ll_loss_weight", 1.0)'
    old_sum = '''        tversky_loss,focal_loss=tversky_da_loss+tversky_ll_loss,focal_da_loss+ focal_ll_loss'''
    new_sum = '''        tversky_loss,focal_loss=tversky_da_loss+self.ll_weight*tversky_ll_loss,focal_da_loss+self.ll_weight*focal_ll_loss'''
    assert old_init in loss_src and old_sum in loss_src, 'loss.py TotalLoss 코드가 예상과 다름'
    loss_src = loss_src.replace(old_init, new_init).replace(old_sum, new_sum)
    with open(loss_path, 'w') as f:
        f.write(loss_src)
    print('loss.py: ll_weight 패치 완료')

with open(loss_path) as f:
    loss_src = f.read()
old_target = '''    target = target.type(output.type())'''
if old_target in loss_src:
    loss_src = loss_src.replace(old_target, '''    target = target.type_as(output)''')
    with open(loss_path, 'w') as f:
        f.write(loss_src)
    print('loss.py: target.type_as 패치 완료')
else:
    print('loss.py: target.type_as 이미 반영됨/해당없음 -> 스킵')

with open(loss_path) as f:
    loss_src = f.read()
old_da = '''        _,seg_da= torch.max(seg_da, 1)
        seg_da=seg_da.cuda()

        _,seg_ll= torch.max(seg_ll, 1)
        seg_ll=seg_ll.cuda()'''
new_da = '''        _,seg_da= torch.max(seg_da, 1)
        seg_da=seg_da.to(out_da.device)

        _,seg_ll= torch.max(seg_ll, 1)
        seg_ll=seg_ll.to(out_ll.device)'''
if old_da in loss_src:
    loss_src = loss_src.replace(old_da, new_da)
    with open(loss_path, 'w') as f:
        f.write(loss_src)
    print('loss.py: seg_da/seg_ll device-agnostic 패치 완료')
else:
    print('loss.py: seg_da/seg_ll 이미 패치됨/해당없음 -> 스킵')

with open(loss_path) as f:
    loss_src2 = f.read()
if '[:,:,12:-12]' not in loss_src2:
    print('loss.py: 크롭 이미 제거됨 -> 스킵')
else:
    loss_src2 = loss_src2.replace('out=outputs[:,:,12:-12]', 'out=outputs')
    loss_src2 = loss_src2.replace('out_da,out_ll=out_da[:,:,12:-12],out_ll[:,:,12:-12]', 'out_da,out_ll=out_da,out_ll')
    with open(loss_path, 'w') as f:
        f.write(loss_src2)
    print('loss.py: 크롭 제거 완료')

utils_path = f'{REPO_DIR}/utils.py'
with open(utils_path) as f:
    utils_src = f.read()
if "lr_mult" in utils_src and "[:,12:-12]" not in utils_src and "args.device" in utils_src:
    print('utils.py: 이미 전부 패치됨 -> 스킵')
else:
    old_sched = '''def poly_lr_scheduler(args, hyp, optimizer, epoch, power=1.5):
    lr = round(hyp['lr'] * (1 - epoch / args.max_epochs) ** power, 8)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    return lr'''
    new_sched = '''def poly_lr_scheduler(args, hyp, optimizer, epoch, power=1.5):
    lr = round(hyp['lr'] * (1 - epoch / args.max_epochs) ** power, 8)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr * param_group.get('lr_mult', 1.0)
    return lr'''
    if old_sched in utils_src:
        utils_src = utils_src.replace(old_sched, new_sched)
    utils_src = utils_src.replace("        da_predict = da_predict[:,12:-12]\n", "")
    utils_src = utils_src.replace("        ll_predict = ll_predict[:,12:-12]\n", "")
    utils_src = utils_src.replace("        predict = predict[:,12:-12]\n", "")
    utils_src = utils_src.replace(
        "        if args.onGPU == True:\n            input = input.cuda().float() / 255.0        \n",
        "        if args.onGPU == True:\n            input = input.to(args.device).float() / 255.0\n")
    utils_src = utils_src.replace(
        "        with torch.cuda.amp.autocast():",
        "        with torch.autocast(device_type=args.device.type, enabled=(args.device.type == 'cuda')):")
    utils_src = utils_src.replace(
        "        input = input.cuda().half() / 255.0 if half else input.cuda().float() / 255.0",
        "        input = input.to(args.device).half() / 255.0 if half else input.to(args.device).float() / 255.0")
    with open(utils_path, 'w') as f:
        f.write(utils_src)
    print('utils.py: device-agnostic 패치 완료')

bdd_path = f'{REPO_DIR}/BDD100K.py'
with open(bdd_path) as f:
    bdd_src = f.read()
if 'letterbox(image' not in bdd_src:
    print('BDD100K.py: 이미 패치됨 -> 스킵')
else:
    bdd_src = bdd_src.replace('image = letterbox(image, (H_, W_))', 'image = cv2.resize(image, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label, (W_, 360))', 'cv2.resize(label, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label1, (W_, 360))', 'cv2.resize(label1, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label2, (W_, 360))', 'cv2.resize(label2, (W_, H_))')
    with open(bdd_path, 'w') as f:
        f.write(bdd_src)
    print('BDD100K.py: 패치 완료 -> letterbox 제거, plain resize(640x384)')

## 4. 파인튜닝 스크립트 작성 (`finetune_ensemble_v2_local_rtx.ipynb`와 동일, device-agnostic `--seed` 지원)

In [ ]:
finetune_script = r'''
import os
import random
import torch
import torch.optim.lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import yaml
import math
from copy import deepcopy
from argparse import ArgumentParser

from model.model import TwinLiteNetPlus
from loss import TotalLoss
from utils import train, val, netParams, save_checkpoint, poly_lr_scheduler
import BDD100K

class ModelEMA:
    def __init__(self, model, decay=0.9999, updates=0):
        self.ema = deepcopy(model).eval()
        self.updates = updates
        self.decay = lambda x: decay * (1 - math.exp(-x / 2000))
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            self.updates += 1
            d = self.decay(self.updates)
            msd = model.state_dict()
            for k, v in self.ema.state_dict().items():
                if v.dtype.is_floating_point:
                    v *= d
                    v += (1. - d) * msd[k].detach()

def resolve_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def train_net(args, hyp):
    use_ema = args.ema
    device = resolve_device()
    args.device = device
    args.onGPU = device.type != 'cpu'
    print(f'[finetune] device: {device}')

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    model = TwinLiteNetPlus(args)

    if args.weight and os.path.isfile(args.weight):
        state = torch.load(args.weight, map_location='cpu')
        if isinstance(state, dict) and 'state_dict' in state:
            state = state['state_dict']
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f'[finetune] pretrained 로드: {args.weight}')
        print(f'[finetune]   missing={len(missing)} unexpected={len(unexpected)}')
    else:
        print(f'[finetune] --weight 없음/파일 없음({args.weight}) - 랜덤 초기화로 진행')

    os.makedirs(args.savedir, exist_ok=True)

    g = torch.Generator()
    g.manual_seed(args.seed)
    trainLoader = torch.utils.data.DataLoader(
        BDD100K.Dataset(hyp, valid=False),
        batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers,
        pin_memory=(device.type == 'cuda'), generator=g)

    valLoader = torch.utils.data.DataLoader(
        BDD100K.Dataset(hyp, valid=True),
        batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers,
        pin_memory=(device.type == 'cuda'))

    model = model.to(device)
    if device.type == 'cuda':
        cudnn.benchmark = True

    print(f'Total network parameters: {netParams(model)}')

    criteria = TotalLoss(hyp)
    start_epoch = 0
    lr = hyp['lr']
    ll_lr_mult = hyp.get('ll_lr_mult', 1.0)
    ll_param_ids = set()
    ll_params = []
    for name, p in model.named_parameters():
        if '_ll' in name:
            ll_params.append(p)
            ll_param_ids.add(id(p))
    other_params = [p for p in model.parameters() if id(p) not in ll_param_ids]
    print(f'[finetune] ll 디코더 파라미터 {len(ll_params)}개 텐서 (LR x{ll_lr_mult}) / 나머지 {len(other_params)}개 텐서')
    optimizer = torch.optim.AdamW([
        {'params': other_params, 'lr': lr, 'lr_mult': 1.0},
        {'params': ll_params, 'lr': lr * ll_lr_mult, 'lr_mult': ll_lr_mult},
    ], betas=(hyp['momentum'], 0.999), eps=hyp['eps'], weight_decay=hyp['weight_decay'])

    ema = ModelEMA(model) if use_ema else None

    best_da_miou = -1.0
    best_ll_iou = -1.0
    if args.resume and os.path.isfile(args.resume):
        if args.resume.endswith('.tar'):
            print(f"=> Loading checkpoint '{args.resume}'")
            checkpoint = torch.load(args.resume, map_location='cpu')
            start_epoch = checkpoint['epoch']
            model.load_state_dict(checkpoint['state_dict'])
            model = model.to(device)
            if use_ema:
                ema.ema.load_state_dict(checkpoint['ema_state_dict'])
                ema.ema = ema.ema.to(device)
                ema.updates = checkpoint['updates']
            optimizer.load_state_dict(checkpoint['optimizer'])
            best_da_miou = checkpoint.get('best_da_miou', -1.0)
            best_ll_iou = checkpoint.get('best_ll_iou', -1.0)
            print(f"=> Loaded checkpoint (epoch {checkpoint['epoch']}, best_da_miou={best_da_miou:.3f}, best_ll_iou={best_ll_iou:.3f})")
        else:
            print(f"=> No valid checkpoint found at '{args.resume}'")

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    for epoch in range(start_epoch, args.max_epochs):
        model_file_name = os.path.join(args.savedir, f'model_{epoch}.pth')
        poly_lr_scheduler(args, hyp, optimizer, epoch)
        lr = optimizer.param_groups[0]['lr']
        ll_lr = optimizer.param_groups[1]['lr'] if len(optimizer.param_groups) > 1 else lr
        print(f'Learning rate: {lr} (ll decoder: {ll_lr})')

        model.train()
        ema = train(args, trainLoader, model, criteria, optimizer, epoch, scaler, args.verbose, ema if use_ema else None)

        model.eval()
        da_segment_results, ll_segment_results = val(valLoader, ema.ema if use_ema else model, args=args)

        print(f'Driving Area Segment: mIOU({da_segment_results[2]:.3f})')
        print(f'Lane Line Segment: Acc({ll_segment_results[0]:.3f}) IOU({ll_segment_results[1]:.3f})')

        torch.save(ema.ema.state_dict(), model_file_name) if use_ema else torch.save(model.state_dict(), model_file_name)
        if da_segment_results[2] > best_da_miou:
            best_da_miou = da_segment_results[2]
            best_path = os.path.join(args.savedir, 'best.pth')
            torch.save(ema.ema.state_dict() if use_ema else model.state_dict(), best_path)
            print(f'[finetune] 새 best(da) 저장: {best_path} (da mIoU={best_da_miou:.3f})')
        if ll_segment_results[1] > best_ll_iou:
            best_ll_iou = ll_segment_results[1]
            best_ll_path = os.path.join(args.savedir, 'best_ll.pth')
            torch.save(ema.ema.state_dict() if use_ema else model.state_dict(), best_ll_path)
            print(f'[finetune] 새 best(ll) 저장: {best_ll_path} (ll IOU={best_ll_iou:.3f})')

        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'ema_state_dict': ema.ema.state_dict() if use_ema else None,
            'updates': ema.updates if use_ema else None,
            'optimizer': optimizer.state_dict(),
            'lr': lr,
            'best_da_miou': best_da_miou,
            'best_ll_iou': best_ll_iou,
        }, os.path.join(args.savedir, 'checkpoint.pth.tar'))

if __name__ == '__main__':
    parser = ArgumentParser()
    parser.add_argument('--max_epochs', type=int, default=100)
    parser.add_argument('--num_workers', type=int, default=4)
    parser.add_argument('--batch_size', type=int, default=8)
    parser.add_argument('--savedir', default='./finetune_out')
    parser.add_argument('--hyp', type=str, default='./hyperparameters/finetune_hyper.yaml')
    parser.add_argument('--resume', type=str, default='')
    parser.add_argument('--weight', type=str, default='')
    parser.add_argument('--config', default='small')
    parser.add_argument('--verbose', action='store_true')
    parser.add_argument('--ema', action='store_true')
    parser.add_argument('--drive_backup_dir', type=str, default='')
    parser.add_argument('--seed', type=int, default=0, help='앙상블 멤버 구분용 시드')
    args = parser.parse_args()

    with open(args.hyp, errors='ignore') as f:
        hyp = yaml.safe_load(f)

    train_net(args, hyp.copy())
'''

with open(f'{REPO_DIR}/finetune.py', 'w') as f:
    f.write(finetune_script)
print('작성 완료:', f'{REPO_DIR}/finetune.py')

## 5. 앙상블 5개 학습 실행 (seed 0~4 순차)

**참고**: 이 세션에서는 노트북이 아니라 동일한 코드를 `~/fine-tune/run_ensemble_v2.sh`
(nohup 백그라운드)로 미리 돌려뒀음 — 1epoch 사니티 테스트로 seed 0에서 da mIoU 0.930/
ll IOU 0.421(1epoch만에), epoch당 ~99초 확인함(40epoch x 5seed 총 5.5시간 예상).
노트북으로 이어서 실행하려면 아래 셀을 그대로 쓰면 됨(멱등 - 이미 완료된 seed는
`best.pth`/`checkpoint.pth.tar` 존재 여부로 건너뛰게 하고 싶으면 조건 추가할 것).

In [ ]:
%cd {REPO_DIR}
MAX_EPOCHS = 40
BATCH_SIZE = 8

for SEED in range(5):
    SAVEDIR = os.path.join(BASE_DIR, f'finetune_out_ensemble_v2_seed{SEED}')
    print(f'=== seed {SEED} 시작 ===')
    train_cmd = (
        f'python finetune.py --config {CONFIG} --weight pretrained/{CONFIG}.pth '
        f'--hyp hyperparameters/finetune_hyper.yaml --max_epochs {MAX_EPOCHS} '
        f'--batch_size {BATCH_SIZE} --savedir "{SAVEDIR}" --seed {SEED} --verbose'
    )
    print(train_cmd)
    get_ipython().system(train_cmd)
    print(f'=== seed {SEED} 완료 ===')

print('앙상블 5개 전부 완료')

## 6. 완료 후 - Mac으로 파일 가져오기

`{BASE_DIR}/finetune_out_ensemble_v2_seed{0..4}/best.pth`(+`best_ll.pth`) 총 10개
파일을 Mac의 `fine-tune/outputs/ensemble_bootstrap_v2/seed{N}/`로 복사(클라우드/USB
등) → `ENSEMBLE_DIR`을 `_v1` -> `_v2`로 바꿔서 기존 평가 스크립트로 재검증(§2.22
"다음 할 일" -3번 참고).